# 定义模型

因为需要识别图片，需要使用 `image` 模型。多模态模型。

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv()

model = init_chat_model(
    model="qwen3.6-35b-a3b",
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)
print('Init ai success!')


# 定义工具



In [ ]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(
    max_results=3,
    topic="general"
)
print('Init TavilySearch success!')

# 添加记忆管理

采用sqlite

In [ ]:
import os
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# create db folder
os.makedirs('../db', exist_ok=True)

connection = sqlite3.connect('../db/class_24.db', check_same_thread=False)
checkpointer = SqliteSaver(connection)
checkpointer.setup()

print('Init SqliteSaver success!')


# 定义智能体

In [ ]:
from langchain.agents import create_agent

system_prompt = """
你是天气助手，你可以回答用户关于天气的问题。
"""

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer,
)

print('Agent is ready.')

# 测试

In [ ]:
response = agent.invoke({"messages": [{"role": "user", "content": "广州今天的天气怎么样" }]}, config={"configurable": {"thread_id": "11"}},)

print('response success~!')

In [ ]:
# 友好打印
for message in response["messages"]:
    message.pretty_print()

print('Happy ending ~!')
